# Data Preparation and Exploratory Analysis

This notebook prepares the January 2024 NYC Yellow Taxi trip data for the multi-scale mobility forecasting analysis. It includes data cleaning, spatial filtering, hourly demand construction and exploratory analysis of taxi demand across Manhattan.

In [ ]:
# Importing the required libraries

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

## Setting the File Paths

In [ ]:
# Setting the project directory

project_dir = Path("..")

# Setting the input paths

raw_taxi_path = (
    project_dir
    / "Data"
    / "Raw"
    / "yellow_tripdata_2024-01.parquet"
)

zone_lookup_path = (
    project_dir
    / "Data"
    / "Spatial"
    / "taxi_zone_lookup.csv"
)

taxi_zones_path = (
    project_dir
    / "Data"
    / "Spatial"
    / "taxi_zones"
    / "taxi_zones.shp"
)

# Setting the processed-data and figure paths

processed_dir = project_dir / "Data" / "Processed"
figures_dir = project_dir / "Output" / "Figures"

In [ ]:
print("Raw taxi data:", raw_taxi_path.exists())
print("Taxi-zone lookup:", zone_lookup_path.exists())
print("Taxi-zone shapefile:", taxi_zones_path.exists())

## Loading the Raw Taxi Data

In [ ]:
# Loading the January 2024 Yellow Taxi trip data

taxi_data = pd.read_parquet(raw_taxi_path)

print("Dataset shape:", taxi_data.shape)
print("Columns:")
print(taxi_data.columns.tolist())

display(taxi_data.head())

## Checking the Raw Dataset

In [ ]:
# Checking the main fields used in the analysis

required_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
]

missing_columns = [
    column for column in required_columns
    if column not in taxi_data.columns
]

print("Missing required columns:", missing_columns)
print("Missing pickup LocationIDs:", taxi_data["PULocationID"].isna().sum())
print("Missing drop-off LocationIDs:", taxi_data["DOLocationID"].isna().sum())

print(
    "Pickup time range:",
    taxi_data["tpep_pickup_datetime"].min(),
    "to",
    taxi_data["tpep_pickup_datetime"].max()
)

## Selecting January 2024 Trips

In [ ]:
# Keeping trips with pickups in January 2024

taxi = taxi_data[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "passenger_count",
        "trip_distance"
    ]
].copy()

taxi = taxi[
    (taxi["tpep_pickup_datetime"] >= "2024-01-01")
    & (taxi["tpep_pickup_datetime"] < "2024-02-01")
].copy()

print("Rows after date filtering:", len(taxi))
print("Pickup start:", taxi["tpep_pickup_datetime"].min())
print("Pickup end:", taxi["tpep_pickup_datetime"].max())

Only trips with pickup times within January 2024 are retained for the study.

## Removing Invalid Trips

In [ ]:
# Removing trips with non-positive distance

taxi = taxi[taxi["trip_distance"] > 0].copy()

# Removing invalid pickup and drop-off zone IDs

taxi = taxi[
    (taxi["PULocationID"] > 0)
    & (taxi["DOLocationID"] > 0)
].copy()

print("Rows after distance and zone checks:", len(taxi))

## Creating Trip Duration

In [ ]:
# Calculating trip duration in minutes

taxi["duration_min"] = (
    taxi["tpep_dropoff_datetime"]
    - taxi["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print(taxi["duration_min"].describe())

## Filtering Trip Duration

Trips with durations below 1 minute or above 180 minutes are removed as unrealistic observations.

In [ ]:
# Keeping trips with realistic durations

taxi = taxi[
    (taxi["duration_min"] >= 1)
    & (taxi["duration_min"] <= 180)
].copy()

print("Rows after duration filtering:", len(taxi))
print(
    "Duration range:",
    taxi["duration_min"].min(),
    "to",
    taxi["duration_min"].max()
)

## Checking the Cleaned Data

In [ ]:
# Checking the cleaned dataset

print("Cleaned dataset shape:", taxi.shape)

print("\nMissing values:")
print(
    taxi[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID",
            "DOLocationID",
            "trip_distance",
            "duration_min"
        ]
    ].isna().sum()
)

In [ ]:
# Checking missing values in the cleaned dataset

print(taxi.isna().sum())

Missing values remain only in `passenger_count`. This variable is not used in the demand forecasting analysis, so these records are retained.

## Saving the Cleaned Dataset

In [ ]:
# Saving the cleaned trip data

cleaned_taxi_path = (
    processed_dir
    / "yellow_taxi_2024_01_cleaned.parquet"
)

taxi.to_parquet(
    cleaned_taxi_path,
    index=False
)

print("Saved:", cleaned_taxi_path)

## Loading the Spatial Data

In [ ]:
# Loading the taxi-zone lookup and shapefile

zone_lookup = pd.read_csv(zone_lookup_path)
zones = gpd.read_file(taxi_zones_path)

print("Zone lookup shape:", zone_lookup.shape)
print("Taxi zones shape:", zones.shape)
print("Taxi zones CRS:", zones.crs)

## Mapping the NYC Study Area

NYC taxi zones are mapped to show the spatial coverage of the dataset. Manhattan is highlighted because the forecasting analysis focuses on the main Manhattan taxi zones.

In [ ]:
# Selecting the main Manhattan taxi zones

excluded_zone_ids = [103, 104, 105, 194, 202]

manhattan_zones = zones[
    (zones["borough"] == "Manhattan")
    & (~zones["LocationID"].isin(excluded_zone_ids))
].copy()

# Creating the study-area map

fig, ax = plt.subplots(figsize=(8, 8))

zones.plot(
    ax=ax,
    color="lightgrey",
    edgecolor="white",
    linewidth=0.4
)

manhattan_zones.plot(
    ax=ax,
    color="orange",
    edgecolor="black",
    linewidth=0.5
)

# Centering the NYC map
minx, miny, maxx, maxy = zones.total_bounds

x_pad = (maxx - minx) * 0.03
y_pad = (maxy - miny) * 0.03

ax.set_xlim(minx - x_pad, maxx + x_pad)
ax.set_ylim(miny - y_pad, maxy + y_pad)
ax.set_aspect("equal")

ax.set_title("NYC Taxi Zones and Manhattan Study Area")
ax.set_axis_off()

plt.tight_layout()

plt.savefig(figures_dir / "nyc_manhattan_study_area.png",dpi=300,bbox_inches="tight")

plt.show()

The forecasting analysis focuses on the main Manhattan taxi zones within the wider NYC taxi-zone network.

## Analysing Pickup Demand by Borough

In [ ]:
# Counting pickups by taxi zone

pickup_counts = (
    taxi
    .groupby("PULocationID")
    .size()
    .reset_index(name="pickup_count")
)

# Adding borough and zone names

pickup_lookup = zone_lookup.rename(
    columns={
        "LocationID": "PULocationID",
        "Borough": "PU_Borough",
        "Zone": "PU_Zone"
    }
)

pickup_counts = pickup_counts.merge(
    pickup_lookup[
        ["PULocationID", "PU_Borough", "PU_Zone"]
    ],
    on="PULocationID",
    how="left"
)

# Filling missing labels

pickup_counts[
    ["PU_Borough", "PU_Zone"]
] = pickup_counts[
    ["PU_Borough", "PU_Zone"]
].fillna("Unknown")

# Summarising pickups by borough

pickup_by_borough = (
    pickup_counts
    .groupby("PU_Borough")["pickup_count"]
    .sum()
    .reset_index()
    .sort_values("pickup_count", ascending=False)
)

display(pickup_by_borough)

Yellow Taxi pickup demand is strongly concentrated in Manhattan, supporting its selection as the main forecasting study area.

## Mapping Pickup Demand Across NYC

In [ ]:
# Preparing the pickup-demand map

pickup_map = zones.merge(
    pickup_counts,
    left_on="LocationID",
    right_on="PULocationID",
    how="left"
)

pickup_map["pickup_count"] = (
    pickup_map["pickup_count"].fillna(0)
)

# Converting to latitude and longitude

pickup_map_plotly = pickup_map.to_crs(epsg=4326)

pickup_map_plotly["LocationID"] = (
    pickup_map_plotly["LocationID"].astype(str)
)

# Creating the map centre

map_center = pickup_map_plotly.geometry.union_all().centroid

# Converting the spatial data to GeoJSON

geojson = pickup_map_plotly.__geo_interface__

In [ ]:
# Creating the pickup-demand map

import plotly.express as px
import warnings

warnings.filterwarnings(
    "ignore",
    message=".*choropleth_mapbox.*is deprecated.*"
)

fig = px.choropleth_mapbox(
    pickup_map_plotly,
    geojson=geojson,
    locations="LocationID",
    featureidkey="properties.LocationID",
    color="pickup_count",
    hover_name="zone",
    hover_data={
        "LocationID": True,
        "borough": True,
        "pickup_count": True
    },
    mapbox_style="carto-positron",
    center={
        "lat": map_center.y,
        "lon": map_center.x
    },
    zoom=10,
    opacity=0.7,
    color_continuous_scale="YlOrRd",
    title="Yellow Taxi Pickup Demand by NYC Taxi Zone - January 2024"
)

fig.update_layout(
    height=700,
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.write_image(figures_dir / "nyc_pickup_demand_jan2024.png",width=1400,height=900,scale=2)

fig.show()

Pickup demand is concentrated mainly in Manhattan, with strong activity also visible around the major airport zones.

## Analysing Drop-off Demand by Borough

In [ ]:
# Counting drop-offs by taxi zone

dropoff_counts = (
    taxi
    .groupby("DOLocationID")
    .size()
    .reset_index(name="dropoff_count")
)

# Adding borough and zone names

dropoff_lookup = zone_lookup.rename(
    columns={
        "LocationID": "DOLocationID",
        "Borough": "DO_Borough",
        "Zone": "DO_Zone"
    }
)

dropoff_counts = dropoff_counts.merge(
    dropoff_lookup[
        ["DOLocationID", "DO_Borough", "DO_Zone"]
    ],
    on="DOLocationID",
    how="left"
)

dropoff_counts[
    ["DO_Borough", "DO_Zone"]
] = dropoff_counts[
    ["DO_Borough", "DO_Zone"]
].fillna("Unknown")

# Summarising drop-offs by borough

dropoff_by_borough = (
    dropoff_counts
    .groupby("DO_Borough")["dropoff_count"]
    .sum()
    .reset_index()
    .sort_values("dropoff_count", ascending=False)
)

display(dropoff_by_borough)

Drop-off demand is also dominated by Manhattan, although destinations are more dispersed across other boroughs than pickup demand.

## Comparing Pickup and Drop-off Demand by Borough

In [ ]:
# Combining pickup and drop-off demand by borough

borough_comparison = pickup_by_borough.merge(
    dropoff_by_borough,
    left_on="PU_Borough",
    right_on="DO_Borough",
    how="outer"
)

borough_comparison["Borough"] = (
    borough_comparison["PU_Borough"]
    .combine_first(borough_comparison["DO_Borough"])
)

borough_comparison = borough_comparison[
    ["Borough", "pickup_count", "dropoff_count"]
].fillna(0)

borough_comparison = borough_comparison.sort_values(
    "pickup_count",
    ascending=False
)

display(borough_comparison)

Manhattan dominates both pickup and drop-off demand, while drop-offs are more widely distributed across other boroughs.

## Identifying the Busiest Pickup Zones

In [ ]:
# Finding the 10 busiest pickup zones

top_pickup_zones = (
    pickup_counts
    .sort_values("pickup_count", ascending=False)
    .head(10)
)

display(
    top_pickup_zones[
        ["PULocationID", "PU_Borough", "PU_Zone", "pickup_count"]
    ]
)

Most of the busiest pickup zones are in Manhattan, while JFK and LaGuardia airports also show high pickup activity.

## Mapping the Busiest Pickup Zones

In [ ]:
# Selecting the top 10 pickup zones

top10_ids = top_pickup_zones["PULocationID"].astype(int).tolist()

# Preparing spatial data

top10_map = pickup_map_plotly.copy()

top10_map["Top_10"] = np.where(
    top10_map["LocationID"].astype(int).isin(top10_ids),
    "Top 10 Pickup Zone",
    "Other Zone"
)

# Creating interactive map

fig = px.choropleth_mapbox(
    top10_map,
    geojson=top10_map.__geo_interface__,
    locations="LocationID",
    featureidkey="properties.LocationID",
    color="Top_10",
    hover_name="zone",
    hover_data={
        "borough": True,
        "pickup_count": ":,",
        "LocationID": True,
        "Top_10": False
    },
    mapbox_style="carto-positron",
    center={
        "lat": map_center.y,
        "lon": map_center.x
    },
    zoom=9.5,
    opacity=0.75,
    title="Top 10 Yellow Taxi Pickup Zones - January 2024"
)

fig.update_layout(
    height=700,
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

The busiest pickup zones are concentrated mainly in Manhattan, particularly around Midtown and the Upper East Side, while JFK and LaGuardia airports also appear among the highest-demand zones.

## Identifying the Busiest Drop-off Zones

In [ ]:
# Finding the 10 busiest drop-off zones

top_dropoff_zones = (
    dropoff_counts
    .sort_values("dropoff_count", ascending=False)
    .head(10)
)

display(
    top_dropoff_zones[
        ["DOLocationID", "DO_Borough", "DO_Zone", "dropoff_count"]
    ]
)

## Mapping the Busiest Drop-off Zones

In [ ]:
# Selecting the top 10 drop-off zones

top10_dropoff_ids = (
    top_dropoff_zones["DOLocationID"]
    .astype(int)
    .tolist()
)

# Preparing spatial data

dropoff_map = zones.merge(
    dropoff_counts,
    left_on="LocationID",
    right_on="DOLocationID",
    how="left"
)

dropoff_map["dropoff_count"] = (
    dropoff_map["dropoff_count"].fillna(0)
)

dropoff_map_plotly = dropoff_map.to_crs(epsg=4326)

dropoff_map_plotly["LocationID"] = (
    dropoff_map_plotly["LocationID"].astype(str)
)

dropoff_map_plotly["Top_10"] = np.where(
    dropoff_map_plotly["LocationID"].astype(int).isin(top10_dropoff_ids),
    "Top 10 Drop-off Zone",
    "Other Zone"
)

# Creating the interactive map

fig = px.choropleth_mapbox(
    dropoff_map_plotly,
    geojson=dropoff_map_plotly.__geo_interface__,
    locations="LocationID",
    featureidkey="properties.LocationID",
    color="Top_10",
    hover_name="zone",
    hover_data={
        "borough": True,
        "dropoff_count": ":,",
        "LocationID": True,
        "Top_10": False
    },
    mapbox_style="carto-positron",
    center={
        "lat": map_center.y,
        "lon": map_center.x
    },
    zoom=9.5,
    opacity=0.75,
    title="Top 10 Yellow Taxi Drop-off Zones - January 2024"
)

fig.update_layout(
    height=700,
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()

 Unlike pickup demand, airport zones do not dominate the busiest drop-off locations. Drop-off demand is more concentrated within Manhattan, indicating different spatial patterns between trip origins and destinations.

## Analysing Hourly Pickup Demand

In [ ]:
# Creating hourly pickup totals across January

hourly_pickups = (
    taxi
    .assign(
        pickup_hour=taxi["tpep_pickup_datetime"].dt.floor("h")
    )
    .groupby("pickup_hour")
    .size()
    .reset_index(name="pickup_count")
)

display(hourly_pickups.head())

In [ ]:
# Visualising hourly pickup demand

fig = px.line(
    hourly_pickups,
    x="pickup_hour",
    y="pickup_count",
    title="Hourly Yellow Taxi Pickup Demand - January 2024",
    labels={
        "pickup_hour": "Time",
        "pickup_count": "Number of Pickups"
    }
)

fig.update_layout(
    height=500,
    hovermode="x unified"
)
fig.write_image(figures_dir / "hourly_pickup_demand_jan2024.png",width=1400,height=800,scale=2)

fig.show()

Yellow Taxi pickup demand varies substantially across time, with repeated daily peaks and lower demand during overnight periods, demonstrating a strong temporal pattern in taxi activity.

## Analysing Average Pickup Demand by Hour

In [ ]:
# Calculating average pickup demand by hour of day

hourly_pickups["hour"] = hourly_pickups["pickup_hour"].dt.hour

average_hourly_demand = (
    hourly_pickups
    .groupby("hour")["pickup_count"]
    .mean()
    .reset_index()
)

display(average_hourly_demand)

In [ ]:
# Visualising average pickup demand by hour of day

fig = px.line(
    average_hourly_demand,
    x="hour",
    y="pickup_count",
    markers=True,
    title="Average Yellow Taxi Pickup Demand by Hour of Day",
    labels={
        "hour": "Hour of Day",
        "pickup_count": "Average Number of Pickups"
    }
)

fig.update_layout(
    height=500,
    xaxis=dict(
        tickmode="linear",
        dtick=1
    )
)
fig.write_image(
    figures_dir / "average_pickup_demand_by_hour.png",
    width=1200,
    height=700,
    scale=2
)
fig.show()

 Average pickup demand follows a clear daily cycle. Demand is lowest during the early morning, increases rapidly after 06:00, and reaches its highest levels during the late afternoon and early evening.

## Analysing Pickup Demand by Day of Week

In [ ]:
# Adding day-of-week information

hourly_pickups["day_of_week"] = (
    hourly_pickups["pickup_hour"].dt.day_name()
)

# Calculating average hourly pickup demand by day

weekday_demand = (
    hourly_pickups
    .groupby("day_of_week")["pickup_count"]
    .mean()
    .reset_index()
)

# Ordering the days correctly

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_demand["day_of_week"] = pd.Categorical(
    weekday_demand["day_of_week"],
    categories=day_order,
    ordered=True
)

weekday_demand = weekday_demand.sort_values("day_of_week")

display(weekday_demand)

In [ ]:
# Visualising average pickup demand by day of week

fig = px.bar(
    weekday_demand,
    x="day_of_week",
    y="pickup_count",
    title="Average Yellow Taxi Pickup Demand by Day of Week",
    labels={
        "day_of_week": "Day of Week",
        "pickup_count": "Average Hourly Pickups"
    }
)

fig.update_layout(
    height=500
)

fig.show()

Average hourly pickup demand varies across the week, with the highest demand occurring on Thursday and relatively high activity on Saturday. Monday and Sunday show lower average demand.

## Creating Hourly Pickup Demand

In [ ]:
# Creating an hourly timestamp by rounding each pickup time down to the nearest hour

taxi["pickup_hour"] = (
    taxi["tpep_pickup_datetime"]
    .dt.floor("h")
)

# Grouping trips by pickup hour and pickup taxi zone
# Each row represents the number of pickups occurring in one zone during one hour

hourly_pickup_demand = (
    taxi
    .groupby(
        ["pickup_hour", "PULocationID"]
    )
    .size()
    .reset_index(name="pickup_count")
)

# Checking the size of the aggregated hourly pickup dataset

print("Shape:", hourly_pickup_demand.shape)

# Displaying the first few rows to verify the structure

display(hourly_pickup_demand.head())

In [ ]:
# Checking that the total number of aggregated pickups

print(
    "Total pickups:",
    hourly_pickup_demand["pickup_count"].sum()
)

## Creating Hourly Destination Demand

In [ ]:
# Creating an hourly timestamp by rounding each drop-off time
# down to the nearest hour

taxi["dropoff_hour"] = (
    taxi["tpep_dropoff_datetime"]
    .dt.floor("h")
)

# Grouping trips by pickup hour and destination taxi zone
# This keeps the drop-off demand table aligned with the same hourly reference used for pickup demand and origin-destination flows

hourly_dropoff_demand = (
    taxi
    .groupby(
        ["pickup_hour", "DOLocationID"]
    )
    .size()
    .reset_index(name="dropoff_count")
)

# Checking the size of the aggregated drop-off dataset

print("Shape:", hourly_dropoff_demand.shape)

# Displaying the first few rows to verify the structure

display(hourly_dropoff_demand.head())

In [ ]:
# Checking that the total number of aggregated drop-offs

print(
    "Total drop-offs:",
    hourly_dropoff_demand["dropoff_count"].sum()
)

## Creating Hourly Origin-Destination Flows


In [ ]:
# Grouping trips by pickup hour, pickup zone, and drop-off zone
# Each row represents the number of taxi trips between one OD pair during a specific hour

hourly_od_flows = (
    taxi
    .groupby(
        ["pickup_hour", "PULocationID", "DOLocationID"]
    )
    .size()
    .reset_index(name="trip_count")
)

# Checking the size of the hourly OD flow dataset

print("Shape:", hourly_od_flows.shape)

# Displaying the first few rows to verify the structure

display(hourly_od_flows.head())

In [ ]:
# Checking that the OD aggregation preserves all cleaned taxi trips

print(
    "Total OD trips:",
    hourly_od_flows["trip_count"].sum()
)

## Adding Taxi Zone Information

In [ ]:
# Creating a lookup table for pickup-zone information

pickup_zone_lookup = zone_lookup.rename(
    columns={
        "LocationID": "PULocationID",
        "Borough": "PU_Borough",
        "Zone": "PU_Zone"
    }
)

# Creating a lookup table for drop-off-zone information

dropoff_zone_lookup = zone_lookup.rename(
    columns={
        "LocationID": "DOLocationID",
        "Borough": "DO_Borough",
        "Zone": "DO_Zone"
    }
)

In [ ]:
# Adding borough and zone names to the hourly pickup-demand dataset

hourly_pickup_demand = hourly_pickup_demand.merge(
    pickup_zone_lookup[
        ["PULocationID", "PU_Borough", "PU_Zone"]
    ],
    on="PULocationID",
    how="left"
)

# Filling unmatched taxi-zone labels with "Unknown"

hourly_pickup_demand[
    ["PU_Borough", "PU_Zone"]
] = hourly_pickup_demand[
    ["PU_Borough", "PU_Zone"]
].fillna("Unknown")

display(hourly_pickup_demand.head())

In [ ]:
# Adding borough and zone names to the hourly drop-off-demand dataset

hourly_dropoff_demand = hourly_dropoff_demand.merge(
    dropoff_zone_lookup[
        ["DOLocationID", "DO_Borough", "DO_Zone"]
    ],
    on="DOLocationID",
    how="left"
)

# Filling unmatched taxi-zone labels with "Unknown"

hourly_dropoff_demand[
    ["DO_Borough", "DO_Zone"]
] = hourly_dropoff_demand[
    ["DO_Borough", "DO_Zone"]
].fillna("Unknown")

display(hourly_dropoff_demand.head())

In [ ]:
# Selecting the OD flow columns required for spatial information

base_columns = [
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "trip_count"
]

hourly_od_flows = hourly_od_flows[base_columns].copy()


# Adding origin borough and taxi-zone information

hourly_od_flows = hourly_od_flows.merge(
    pickup_zone_lookup[
        ["PULocationID", "PU_Borough", "PU_Zone"]
    ],
    on="PULocationID",
    how="left"
)


# Adding destination borough and taxi-zone information

hourly_od_flows = hourly_od_flows.merge(
    dropoff_zone_lookup[
        ["DOLocationID", "DO_Borough", "DO_Zone"]
    ],
    on="DOLocationID",
    how="left"
)


# Replacing unmatched zone labels with "Unknown"

hourly_od_flows[
    ["PU_Borough", "PU_Zone", "DO_Borough", "DO_Zone"]
] = hourly_od_flows[
    ["PU_Borough", "PU_Zone", "DO_Borough", "DO_Zone"]
].fillna("Unknown")


# Checking the resulting OD flow dataset

print("Shape:", hourly_od_flows.shape)

display(hourly_od_flows.head())

## Creating Temporal Features

In [ ]:
# Extracting calendar-based temporal features from the hourly timestamp

hourly_od_flows["hour"] = (
    hourly_od_flows["pickup_hour"].dt.hour
)

hourly_od_flows["day_of_week"] = (
    hourly_od_flows["pickup_hour"].dt.dayofweek
)

hourly_od_flows["month"] = (
    hourly_od_flows["pickup_hour"].dt.month
)


# Identifying weekend observations
# Monday = 0 and Sunday = 6

hourly_od_flows["is_weekend"] = (
    hourly_od_flows["day_of_week"] >= 5
).astype(int)


# Checking the newly created temporal features

display(
    hourly_od_flows[
        [
            "pickup_hour",
            "hour",
            "day_of_week",
            "is_weekend",
            "month"
        ]
    ].head()
)

## Adding Service-Zone Information

In [ ]:
# Preparing the hourly OD dataset with spatial and temporal information

# Keeping the core hourly OD-flow variables

hourly_od_flows = hourly_od_flows[
    [
        "pickup_hour",
        "PULocationID",
        "DOLocationID",
        "trip_count"
    ]
].copy()


# Creating basic calendar features from the hourly timestamp

hourly_od_flows["hour"] = (
    hourly_od_flows["pickup_hour"].dt.hour)

hourly_od_flows["day"] = (
    hourly_od_flows["pickup_hour"].dt.day)

hourly_od_flows["day_of_week"] = (
    hourly_od_flows["pickup_hour"].dt.dayofweek)

hourly_od_flows["month"] = (
    hourly_od_flows["pickup_hour"].dt.month)

hourly_od_flows["is_weekend"] = (
    hourly_od_flows["day_of_week"]
    .isin([5, 6])
    .astype(int))


# Preparing pickup-zone information

pickup_zone_info = zone_lookup[
    [
        "LocationID",
        "Borough",
        "Zone",
        "service_zone"
    ]
].rename(
    columns={
        "LocationID": "PULocationID",
        "Borough": "PU_Borough",
        "Zone": "PU_Zone",
        "service_zone": "PU_service_zone"
    }
)


# Preparing drop-off-zone information

dropoff_zone_info = zone_lookup[
    [
        "LocationID",
        "Borough",
        "Zone",
        "service_zone"
    ]
].rename(
    columns={
        "LocationID": "DOLocationID",
        "Borough": "DO_Borough",
        "Zone": "DO_Zone",
        "service_zone": "DO_service_zone"
    }
)


# Adding pickup-zone information

hourly_od_flows = hourly_od_flows.merge(
    pickup_zone_info,
    on="PULocationID",
    how="left"
)


# Adding drop-off-zone information

hourly_od_flows = hourly_od_flows.merge(
    dropoff_zone_info,
    on="DOLocationID",
    how="left"
)


# Filling unmatched spatial labels

spatial_columns = [
    "PU_Borough",
    "PU_Zone",
    "PU_service_zone",
    "DO_Borough",
    "DO_Zone",
    "DO_service_zone"
]

hourly_od_flows[spatial_columns] = (
    hourly_od_flows[spatial_columns]
    .fillna("Unknown")
)


# Arranging the columns consistently

hourly_od_flows = hourly_od_flows[
    [
        "pickup_hour",
        "PULocationID",
        "DOLocationID",
        "trip_count",
        "hour",
        "day",
        "day_of_week",
        "month",
        "is_weekend",
        "PU_Borough",
        "PU_Zone",
        "PU_service_zone",
        "DO_Borough",
        "DO_Zone",
        "DO_service_zone"
    ]
]


# Checking the resulting dataset

print("Shape:", hourly_od_flows.shape)

display(hourly_od_flows.head())

## Checking the Processed Hourly Datasets

In [ ]:
# Checking the final dimensions of the processed hourly datasets

print("Hourly pickup demand shape:", hourly_pickup_demand.shape)
print("Hourly drop-off demand shape:", hourly_dropoff_demand.shape)
print("Hourly OD-flow shape:", hourly_od_flows.shape)


# Checking that total demand is preserved across all aggregations

print(
    "\nTotal pickup demand:",
    hourly_pickup_demand["pickup_count"].sum()
)

print(
    "Total drop-off demand:",
    hourly_dropoff_demand["dropoff_count"].sum()
)

print(
    "Total OD-flow demand:",
    hourly_od_flows["trip_count"].sum()
)


# Checking the temporal coverage of the OD dataset

print(
    "\nOD start time:",
    hourly_od_flows["pickup_hour"].min()
)

print(
    "OD end time:",
    hourly_od_flows["pickup_hour"].max()
)

print(
    "Unique hourly timestamps:",
    hourly_od_flows["pickup_hour"].nunique()
)


# Checking duplicate hourly OD records

duplicate_od_records = hourly_od_flows.duplicated(
    subset=[
        "pickup_hour",
        "PULocationID",
        "DOLocationID"
    ]
).sum()

print(
    "\nDuplicate hourly OD records:",
    duplicate_od_records
)

## Saving the Processed Hourly Datasets

In [ ]:
# Defining output paths for the processed hourly datasets

hourly_pickup_path = (
    processed_dir / "hourly_pickup_demand_2024_01.parquet"
)

hourly_dropoff_path = (
    processed_dir / "hourly_dropoff_demand_2024_01.parquet"
)

hourly_od_path = (
    processed_dir / "hourly_od_flows_2024_01.parquet"
)


# Saving the hourly pickup-demand dataset

hourly_pickup_demand.to_parquet(
    hourly_pickup_path,
    index=False
)


# Saving the hourly drop-off-demand dataset

hourly_dropoff_demand.to_parquet(
    hourly_dropoff_path,
    index=False
)


# Saving the hourly OD-flow dataset

hourly_od_flows.to_parquet(
    hourly_od_path,
    index=False
)


# Confirming the saved output files

print("Saved:", hourly_pickup_path)
print("Saved:", hourly_dropoff_path)
print("Saved:", hourly_od_path)

## Verifying the Saved Datasets

In [ ]:
# Reloading the saved processed datasets

pickup_check = pd.read_parquet(hourly_pickup_path)
dropoff_check = pd.read_parquet(hourly_dropoff_path)
od_check = pd.read_parquet(hourly_od_path)


# Checking the reloaded dataset shapes

print("Reloaded pickup shape:", pickup_check.shape)
print("Reloaded drop-off shape:", dropoff_check.shape)
print("Reloaded OD-flow shape:", od_check.shape)


# Checking that the total demand remains unchanged

print(
    "\nReloaded pickup total:",
    pickup_check["pickup_count"].sum()
)

print(
    "Reloaded drop-off total:",
    dropoff_check["dropoff_count"].sum()
)

print(
    "Reloaded OD-flow total:",
    od_check["trip_count"].sum()
)